# 2. The shape of the market

Characterise the target variable before modelling it.
This notebook develops the **stylised facts** of SA1 electricity
prices — heavy tails, intra-day seasonality, volatility clustering,
and the price-duration curve — that any forecasting model must respect.

## Objectives

- Plot the empirical distribution of half-hourly prices: raw histogram, asinh-transformed histogram, QQ plot.
- Decompose temporal profiles: hour-of-day, day-of-week, month-of-year via heatmaps and boxplots.
- Compute autocorrelation and partial autocorrelation — understand the lag-48 spike.
- Visualise rolling volatility and volatility clustering.
- Build a price-duration curve.
- Map negative-price and spike frequency across all five regions.
- Quantify heavy tails: excess kurtosis and a simple tail index.
- Summarise at least six stylised facts with saved figures.

## Prerequisites

- Notebook 01 completed — cached parquet files at `data/processed/SA1_5min.parquet` and `SA1_30min.parquet`.

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

from grian.config import load_config, repo_root
from grian.data import load_prices
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
warnings.filterwarnings("ignore", category=FutureWarning)

REGION = cfg["region"]
spike_threshold = cfg["spike_threshold_aud"]

REGION_COLORS = {
    "NSW1": "#2196F3", "QLD1": "#FF9800", "VIC1": "#4CAF50",
    "SA1": "#F44336", "TAS1": "#9C27B0",
}

---
## 1. Load the SA1 dataset

We work with the 5-minute dispatch prices from notebook 01's cached
parquet. For ACF analysis we'll also use the 30-minute trading prices
(one lag = one half-hour, so lag-48 = same time yesterday).

In [ ]:
processed = Path(cfg["paths"]["processed"])

df_5min = pd.read_parquet(processed / f"{REGION}_5min.parquet")
df_30min = pd.read_parquet(processed / f"{REGION}_30min.parquet")

print(f"5-min:  {df_5min.shape}  [{df_5min.index[0]} … {df_5min.index[-1]}]")
print(f"30-min: {df_30min.shape} [{df_30min.index[0]} … {df_30min.index[-1]}]")

df_5min.head()

In [ ]:
price = df_5min["price"].dropna()
price_30 = df_30min["price"].dropna()

price.describe()

---
## 2. Price distribution: raw vs asinh-transformed

The raw price distribution is dominated by the bulk near \$50–\$100.
Extreme spikes (up to the market price cap at \$17,500) and negative
prices make a standard histogram almost useless on a linear axis.

The **inverse hyperbolic sine** (`arcsinh`) is our target transform for
modelling. Like `log`, it compresses large values, but unlike `log` it
handles zero and negative inputs. Let's see what the distribution looks
like before and after.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw histogram (clipped)
axes[0].hist(price.clip(-200, 600), bins=400, edgecolor="none", alpha=0.7)
axes[0].set_xlabel("Price ($/MWh)")
axes[0].set_ylabel("Count")
axes[0].set_title("Raw price (clipped to [-200, 600])")
axes[0].axvline(0, color="red", linewidth=0.8, linestyle="--")

# asinh-transformed
price_asinh = np.arcsinh(price)
axes[1].hist(price_asinh, bins=300, edgecolor="none", alpha=0.7, color="C1")
axes[1].set_xlabel("arcsinh(price)")
axes[1].set_ylabel("Count")
axes[1].set_title("arcsinh-transformed price")
axes[1].axvline(0, color="red", linewidth=0.8, linestyle="--")

# QQ plot against normal
stats.probplot(price_asinh.values, dist="norm", plot=axes[2])
axes[2].set_title("QQ plot — arcsinh(price) vs Normal")

fig.tight_layout()
save_fig(fig, "02_price_distribution_raw_vs_asinh")
plt.show()

The transform makes the distribution much closer to symmetric and
bell-shaped, but the QQ plot still shows departures in both tails.
This market is *not* Gaussian — even after the transform, extreme
events occur far more often than a normal distribution predicts.

---
## 3. Temporal profiles

Electricity prices are strongly shaped by when people use power.
Three cycles dominate:

- **Intra-day** — the 24-hour demand cycle (low overnight, morning ramp,
  midday solar dip, evening peak).
- **Weekly** — weekdays vs weekends (commercial/industrial load drops).
- **Seasonal** — summer air conditioning, winter heating, and the solar
  production curve.

### 3a. Hour-of-day boxplots

In [ ]:
df_plot = df_5min[["price"]].dropna().copy()
df_plot["hour"] = df_plot.index.hour
df_plot["dow"] = df_plot.index.dayofweek
df_plot["month"] = df_plot.index.month

fig, ax = plt.subplots(figsize=(14, 5))
# Clip for readable boxplots
df_plot["price_clipped"] = df_plot["price"].clip(-100, 500)
df_plot.boxplot(column="price_clipped", by="hour", ax=ax,
                showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="C0", alpha=0.5))
ax.set_title(f"{REGION} — price by hour of day", fontsize=13)
ax.set_xlabel("Hour")
ax.set_ylabel("Price ($/MWh, clipped)")
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
fig.suptitle("")
fig.tight_layout()
save_fig(fig, "02_price_by_hour")
plt.show()

The midday dip (hours 10–14) is the **solar duck curve** — rooftop
and utility solar push net demand down, collapsing prices. The evening
peak (hours 17–20) is when solar drops off but demand stays high.

### 3b. Hour × month heatmap

In [ ]:
# Median price by hour and month
heatmap_data = df_plot.groupby(["month", "hour"])["price"].median().unstack("hour")

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdYlBu_r",
               vmin=-20, vmax=200)
ax.set_xticks(range(24))
ax.set_xticklabels(range(24))
ax.set_yticks(range(12))
ax.set_yticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_xlabel("Hour of day")
ax.set_ylabel("Month")
ax.set_title(f"{REGION} — median price by hour × month ($/MWh)")
plt.colorbar(im, ax=ax, label="$/MWh", shrink=0.8)
fig.tight_layout()
save_fig(fig, "02_price_heatmap_hour_month")
plt.show()

Two hot zones: summer evenings (Jan–Feb, hours 17–20) and winter
mornings/evenings (Jun–Aug). The cold band across the middle of the day
is solar suppression — strongest in the sunny months.

### 3c. Day-of-week

In [ ]:
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(10, 5))
df_plot["price_clipped"].groupby(df_plot["dow"]).apply(
    lambda x: ax.boxplot(x.values, positions=[x.name], widths=0.6,
                         showfliers=False, patch_artist=True,
                         boxprops=dict(facecolor="C2", alpha=0.5))
)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.set_ylabel("Price ($/MWh, clipped)")
ax.set_title(f"{REGION} — price by day of week")
fig.tight_layout()
save_fig(fig, "02_price_by_dow")
plt.show()

Weekends are lower on average — commercial and industrial demand drops.
But weekend spikes still happen when the system is tight.

### 3d. Monthly boxplots

In [ ]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(14, 5))
df_plot.boxplot(column="price_clipped", by="month", ax=ax,
                showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="C3", alpha=0.5))
ax.set_title(f"{REGION} — price by month", fontsize=13)
ax.set_xticklabels(month_names)
ax.set_xlabel("Month")
ax.set_ylabel("Price ($/MWh, clipped)")
fig.suptitle("")
fig.tight_layout()
save_fig(fig, "02_price_by_month")
plt.show()

---
## 4. Autocorrelation

If prices are autocorrelated, past prices contain information about
future prices — and that information is free to use as a feature.

We use the 30-minute trading price for this analysis (one lag = 30 min,
so lag 48 = 24 hours = same time yesterday).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(price_30.values, lags=336, ax=ax1, alpha=0.05,
         title=f"{REGION} — ACF (30-min trading price)")
ax1.set_xlabel("Lag (half-hours)")
# Mark key lags
for lag, label in [(48, "24h"), (96, "48h"), (336, "7d")]:
    ax1.axvline(lag, color="red", linewidth=0.5, linestyle=":", alpha=0.7)

plot_pacf(price_30.values, lags=100, ax=ax2, alpha=0.05, method="ywm",
          title=f"{REGION} — PACF (30-min trading price)")
ax2.set_xlabel("Lag (half-hours)")
ax2.axvline(48, color="red", linewidth=0.5, linestyle=":", alpha=0.7)

fig.tight_layout()
save_fig(fig, "02_acf_pacf")
plt.show()

The ACF decays slowly with strong spikes at **lag 48** (same time
yesterday) and **lag 336** (same time last week). The PACF shows that
lag 48 has the strongest *direct* relationship after controlling for
intermediate lags.

This tells us: the best single predictor of today's 3pm price is
yesterday's 3pm price. The weekly cycle adds information beyond the
daily cycle. Both should be features in any forecast model.

---
## 5. Rolling volatility and volatility clustering

Electricity price volatility is not constant — it comes in bursts.
A quiet week can be followed by days of extreme swings. This
**volatility clustering** is a hallmark of energy markets.

In [ ]:
# 7-day rolling standard deviation of 30-min prices
rolling_std = price_30.rolling(window=7 * 48, min_periods=48).std()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(price_30.index, price_30.values, linewidth=0.2, alpha=0.5)
ax1.set_ylabel("Price ($/MWh)")
ax1.set_yscale("symlog", linthresh=100)
ax1.set_title(f"{REGION} — 30-min trading price")

ax2.plot(rolling_std.index, rolling_std.values, linewidth=0.8, color="C3")
ax2.set_ylabel("7-day rolling σ ($/MWh)")
ax2.set_title("Rolling volatility (7-day window)")
ax2.set_yscale("log")

fig.tight_layout()
save_fig(fig, "02_rolling_volatility")
plt.show()

The volatility plot shows clear clustering: quiet periods (σ ≈ \$10–30)
interrupted by bursts where σ jumps to \$500+. These bursts often
coincide with heatwaves, generator outages, or wind droughts.

---
## 6. Price-duration curve

A price-duration curve sorts all prices from highest to lowest and
plots them against the fraction of time. It answers: what price level
is exceeded X% of the time?

In [ ]:
sorted_price = np.sort(price.values)[::-1]
pct = np.linspace(0, 100, len(sorted_price))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full range (symlog)
ax1.plot(pct, sorted_price, linewidth=0.5)
ax1.set_yscale("symlog", linthresh=100)
ax1.set_xlabel("% of time price is at or above")
ax1.set_ylabel("Price ($/MWh)")
ax1.set_title("Price-duration curve (full range)")
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.axhline(spike_threshold, color="red", linewidth=0.8, linestyle="--",
            label=f"${spike_threshold} spike threshold")
ax1.legend()

# Zoom into the middle 90%
mask = (pct >= 5) & (pct <= 95)
ax2.plot(pct[mask], sorted_price[mask], linewidth=0.8)
ax2.set_xlabel("% of time price is at or above")
ax2.set_ylabel("Price ($/MWh)")
ax2.set_title("Price-duration curve (5th–95th percentile)")

fig.tight_layout()
save_fig(fig, "02_price_duration_curve")
plt.show()

print(f"Price below $50 for {(price < 50).mean():.1%} of the time")
print(f"Price below $0 for {(price < 0).mean():.1%} of the time")
print(f"Price above ${spike_threshold} for {(price > spike_threshold).mean():.2%} of the time")
print(f"Price above $1000 for {(price > 1000).mean():.3%} of the time")

Most of the time, prices sit in a narrow band. But the tails are where
the money is — a battery that captures even a fraction of the spikes
generates outsized revenue.

---
## 7. Negative prices and spikes — regional comparison

How does SA1's spike and negative-price frequency compare across the
NEM? This matters for deciding whether a model trained on SA1 would
generalise to other regions.

In [ ]:
ALL_REGIONS = ["NSW1", "QLD1", "VIC1", "SA1", "TAS1"]
START = cfg["train_start"]
END = cfg["test_end"]

all_prices = {}
for r in ALL_REGIONS:
    all_prices[r] = load_prices(r, START, END, cache=cfg["nemosis_cache"])

region_stats = []
for r in ALL_REGIONS:
    p = all_prices[r]["price"].dropna()
    region_stats.append({
        "Region": r,
        "% Negative": 100 * (p < 0).mean(),
        f"% Spikes (>${spike_threshold})": 100 * (p > spike_threshold).mean(),
        "Mean ($/MWh)": p.mean(),
        "Std ($/MWh)": p.std(),
        "Kurtosis": p.kurtosis(),
    })

stats_df = pd.DataFrame(region_stats).set_index("Region")
stats_df.round(2)

In [ ]:
# Map: regions coloured by spike frequency, annotated with negative-price share
regions_gdf = gpd.read_file(repo_root() / "data" / "nem_regions.geojson")
regions_plot = regions_gdf.merge(stats_df, left_on="nem_region", right_index=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))

# Spike frequency map
regions_plot.plot(ax=ax1, column=f"% Spikes (>${spike_threshold})", cmap="YlOrRd",
                  edgecolor="white", linewidth=1.5, legend=True,
                  legend_kwds={"label": f"% intervals > ${spike_threshold}", "shrink": 0.5})
for _, row in regions_plot.iterrows():
    ax1.annotate(f"{row['nem_region']}\n{row[f'% Spikes (>${spike_threshold})']:.2f}%",
                 xy=(row["label_lon"], row["label_lat"]), ha="center", fontsize=8,
                 fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax1.set_xlim(112, 155)
ax1.set_ylim(-45, -10)
ax1.set_title(f"Spike frequency (> ${spike_threshold}/MWh)")

# Negative price frequency map
regions_plot.plot(ax=ax2, column="% Negative", cmap="YlGnBu",
                  edgecolor="white", linewidth=1.5, legend=True,
                  legend_kwds={"label": "% negative price intervals", "shrink": 0.5})
for _, row in regions_plot.iterrows():
    ax2.annotate(f"{row['nem_region']}\n{row['% Negative']:.1f}% neg",
                 xy=(row["label_lon"], row["label_lat"]), ha="center", fontsize=8,
                 fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax2.set_xlim(112, 155)
ax2.set_ylim(-45, -10)
ax2.set_title("Negative price frequency")

fig.tight_layout()
save_fig(fig, "02_spike_negative_regional_map")
plt.show()

SA1 and QLD1 lead on both negative prices (high renewable penetration)
and spikes (tight supply when renewables drop). TAS1 is different —
hydro-dominated, fewer extremes in both directions.

---
## 8. Heavy tails — excess kurtosis and "5-sigma" events

A Gaussian distribution has kurtosis 3 (excess kurtosis 0). How does
the price distribution compare? And how many events that *would be*
5-sigma under a Gaussian actually occur?

In [ ]:
# Excess kurtosis
kurt = price.kurtosis()
skew = price.skew()
print(f"Excess kurtosis: {kurt:.1f}  (Gaussian = 0)")
print(f"Skewness: {skew:.2f}")

# "5-sigma" events under a Gaussian assumption
z = (price - price.mean()) / price.std()
for sigma in [3, 4, 5, 6]:
    n_events = (z.abs() > sigma).sum()
    expected = len(z) * 2 * stats.norm.sf(sigma)
    print(f"  |z| > {sigma}: {n_events:>6,} actual vs {expected:>8.1f} expected (Gaussian)  "
          f"— {n_events / max(expected, 1):.0f}× more")

In [ ]:
# Tail index: simple Hill estimator on the upper tail
upper_tail = np.sort(price[price > 0].values)[::-1]
k = min(500, len(upper_tail) // 10)
log_excesses = np.log(upper_tail[:k]) - np.log(upper_tail[k])
hill_alpha = 1.0 / np.mean(log_excesses)
print(f"\nHill tail index (upper tail, k={k}): α = {hill_alpha:.2f}")
print("  α < 2 means infinite variance; α < 1 means infinite mean.")
print("  Typical range for electricity prices: 1.5–4.0")

The excess kurtosis is enormous — hundreds of times the Gaussian value.
"5-sigma events" that should occur once in a million observations happen
routinely. This is why:

1. MSE is a bad loss function for raw prices (it's dominated by spikes).
2. The arcsinh transform helps, but even transformed prices have heavier
   tails than Gaussian.
3. Probabilistic forecasts (quantiles, intervals) are essential — a point
   forecast alone cannot capture the risk.

---
## 9. Stylised facts summary

Every model we build must be consistent with these facts. If a model
produces a forecast that violates them (e.g. never predicts negative
prices, or underestimates spike frequency), something is wrong.

1. **Heavy tails.** Price has extreme excess kurtosis. Spikes to
   \$5,000+ and drops to -\$1,000 occur regularly. The arcsinh
   transform compresses but doesn't eliminate them.

2. **Strong diurnal seasonality.** Prices follow a 24-hour cycle
   driven by demand and solar production. The evening peak (17–20h)
   and midday solar dip are the dominant features.

3. **Weekly pattern.** Weekday prices are higher than weekends on
   average, due to commercial/industrial load.

4. **Lag-48 autocorrelation.** The strongest predictor of today's
   price at hour H is yesterday's price at hour H.

5. **Volatility clustering.** Quiet periods and volatile periods
   alternate — tomorrow's volatility is correlated with today's.

6. **Asymmetric tails.** The right tail (spikes) is heavier than
   the left tail (negative prices), though both are extreme by
   normal-distribution standards.

7. **Regional heterogeneity.** SA1 is more volatile than the eastern
   states. TAS1 (hydro) is structurally different. Models should
   not assume cross-region transferability without testing.

---
## Exercises

### Exercise 1: Volatility clustering — is tomorrow's volatility predictable?

Compute the autocorrelation of **absolute returns** (|p_t - p_{t-1}|)
at the 30-minute frequency. If volatility clusters, absolute returns
should be autocorrelated even if returns themselves are not.

<details><summary>Hint 1</summary>

Compute returns as `price_30.diff()`, then take the absolute value.
Use `plot_acf` from statsmodels on the `.dropna()` result.

</details>

<details><summary>Hint 2</summary>

Compare the ACF of raw returns vs absolute returns. Raw returns
should decay quickly (prices are hard to predict). Absolute returns
should stay correlated much longer (volatility is persistent).

</details>

<details><summary>Hint 3</summary>

If the ACF of absolute returns is still significant at lag 48 (one
day), that means today's volatility predicts tomorrow's. This is
useful for a model — you could add a "recent volatility" feature.

</details>

<details><summary>Solution</summary>

```python
returns = price_30.diff().dropna()
abs_returns = returns.abs()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(returns.values, lags=200, ax=ax1, alpha=0.05,
         title="ACF of returns (price changes)")
ax1.set_xlabel("Lag (half-hours)")

plot_acf(abs_returns.values, lags=200, ax=ax2, alpha=0.05,
         title="ACF of |returns| (volatility proxy)")
ax2.set_xlabel("Lag (half-hours)")

fig.tight_layout()
plt.show()

print(f"ACF of returns at lag 48: {returns.autocorr(lag=48):.3f}")
print(f"ACF of |returns| at lag 48: {abs_returns.autocorr(lag=48):.3f}")
print(f"ACF of |returns| at lag 336: {abs_returns.autocorr(lag=336):.3f}")
```

Raw returns show weak autocorrelation (prices are hard to predict
directionally), but absolute returns remain significantly correlated
for days — volatility today predicts volatility tomorrow. This is
the classic GARCH-type effect. For our forecasting models, a
"rolling volatility" feature (e.g. 48-period rolling std of returns)
would capture this signal.

</details>

In [ ]:
# Your analysis here
returns = price_30.diff().dropna()
abs_returns = returns.abs()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(returns.values, lags=200, ax=ax1, alpha=0.05,
         title="ACF of returns (price changes)")
ax1.set_xlabel("Lag (half-hours)")

plot_acf(abs_returns.values, lags=200, ax=ax2, alpha=0.05,
         title="ACF of |returns| (volatility proxy)")
ax2.set_xlabel("Lag (half-hours)")

fig.tight_layout()
plt.show()

print(f"ACF of returns at lag 48: {returns.autocorr(lag=48):.3f}")
print(f"ACF of |returns| at lag 48: {abs_returns.autocorr(lag=48):.3f}")
print(f"ACF of |returns| at lag 336: {abs_returns.autocorr(lag=336):.3f}")


### Exercise 2: Seasonal decomposition

Use `statsmodels.tsa.seasonal.seasonal_decompose` on the **daily mean
price** to decompose it into trend, seasonal, and residual components.
What is the dominant seasonal period?

<details><summary>Hint 1</summary>

First resample to daily: `daily = price_30.resample('D').mean()`.
Then call `seasonal_decompose(daily.dropna(), model='additive', period=365)`
to decompose with an annual cycle. Try `period=7` for the weekly cycle too.

</details>

<details><summary>Hint 2</summary>

The `seasonal` component shows the repeating pattern. Look at its
amplitude — is the weekly cycle or the annual cycle stronger? The
`resid` component is what the model must explain after removing
these regular patterns.

</details>

<details><summary>Hint 3</summary>

Plot both decompositions (weekly and annual) side by side. The weekly
seasonal component should show a clear Mon–Sun pattern; the annual
should show summer/winter variation.

</details>

<details><summary>Solution</summary>

```python
daily = price_30.resample("D").mean().dropna()

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for col, period, label in [(0, 7, "Weekly"), (1, 365, "Annual")]:
    decomp = seasonal_decompose(daily, model="additive", period=period)
    axes[0, col].plot(decomp.observed, linewidth=0.5)
    axes[0, col].set_title(f"{label} — Observed")
    axes[1, col].plot(decomp.trend, linewidth=0.8)
    axes[1, col].set_title(f"{label} — Trend")
    axes[2, col].plot(decomp.seasonal, linewidth=0.5)
    axes[2, col].set_title(f"{label} — Seasonal (period={period})")
    axes[3, col].plot(decomp.resid, linewidth=0.5)
    axes[3, col].set_title(f"{label} — Residual")

    seasonal_amp = decomp.seasonal.max() - decomp.seasonal.min()
    resid_std = decomp.resid.dropna().std()
    print(f"{label} decomposition:")
    print(f"  Seasonal amplitude: ${seasonal_amp:.1f}/MWh")
    print(f"  Residual std: ${resid_std:.1f}/MWh")
    print()

fig.tight_layout()
plt.show()
```

The weekly cycle has a clear 7-day pattern with lower weekend prices.
The annual cycle shows higher prices in summer and winter with a dip
in the shoulder seasons. The residual still has large spikes — these
are the events driven by weather, outages, and demand shocks that no
simple seasonal model can capture. The residual standard deviation
is much larger than the seasonal amplitude, confirming that most
price variation is non-seasonal.

</details>

In [ ]:
# Your analysis here
daily = price_30.resample("D").mean().dropna()

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for col, period, label in [(0, 7, "Weekly"), (1, 365, "Annual")]:
    decomp = seasonal_decompose(daily, model="additive", period=period)
    axes[0, col].plot(decomp.observed, linewidth=0.5)
    axes[0, col].set_title(f"{label} — Observed")
    axes[1, col].plot(decomp.trend, linewidth=0.8)
    axes[1, col].set_title(f"{label} — Trend")
    axes[2, col].plot(decomp.seasonal, linewidth=0.5)
    axes[2, col].set_title(f"{label} — Seasonal (period={period})")
    axes[3, col].plot(decomp.resid, linewidth=0.5)
    axes[3, col].set_title(f"{label} — Residual")

    seasonal_amp = decomp.seasonal.max() - decomp.seasonal.min()
    resid_std = decomp.resid.dropna().std()
    print(f"{label} decomposition:")
    print(f"  Seasonal amplitude: ${seasonal_amp:.1f}/MWh")
    print(f"  Residual std: ${resid_std:.1f}/MWh")
    print()

fig.tight_layout()
plt.show()


### Exercise 3: Tail behaviour — SA1 vs TAS1

SA1 is wind/solar dominated; TAS1 is hydro dominated. Compare their
tail behaviour. Which has heavier tails? Which has more negative
prices? What does this tell you about the role of generation mix in
price risk?

<details><summary>Hint 1</summary>

Plot the price-duration curves for both regions on the same axes.
Use `np.sort(p.values)[::-1]` for each and align by percentile.

</details>

<details><summary>Hint 2</summary>

Compare the QQ plots: `stats.probplot(np.arcsinh(p), plot=ax)` for
each region. The region with heavier tails will deviate more from
the straight line at the extremes.

</details>

<details><summary>Hint 3</summary>

Think about *why* hydro prices behave differently. Hydro generators
can ramp quickly and have flexible output — they act as a buffer
against extreme prices in both directions. Wind and solar are
price-takers and can't respond to scarcity.

</details>

<details><summary>Solution</summary>

```python
sa1 = all_prices["SA1"]["price"].dropna()
tas1 = all_prices["TAS1"]["price"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price-duration curves
for p, r, c in [(sa1, "SA1", REGION_COLORS["SA1"]),
                (tas1, "TAS1", REGION_COLORS["TAS1"])]:
    sorted_p = np.sort(p.values)[::-1]
    pcts = np.linspace(0, 100, len(sorted_p))
    axes[0].plot(pcts, sorted_p, linewidth=0.8, color=c, label=r)
axes[0].set_yscale("symlog", linthresh=100)
axes[0].set_xlabel("% of time at or above")
axes[0].set_ylabel("Price ($/MWh)")
axes[0].set_title("Price-duration curves")
axes[0].legend()

# QQ plots
stats.probplot(np.arcsinh(sa1.values), dist="norm", plot=axes[1])
axes[1].set_title("QQ plot — SA1 arcsinh(price)")
stats.probplot(np.arcsinh(tas1.values), dist="norm", plot=axes[2])
axes[2].set_title("QQ plot — TAS1 arcsinh(price)")

fig.tight_layout()
plt.show()

for r, p in [("SA1", sa1), ("TAS1", tas1)]:
    print(f"{r}: kurtosis={p.kurtosis():.0f}, skew={p.skew():.1f}, "
          f"{(p < 0).mean():.2%} negative, "
          f"{(p > spike_threshold).mean():.3%} spikes")
```

SA1 has dramatically heavier tails than TAS1 in both directions.
TAS1's hydro fleet provides a natural buffer — generators ramp up
during scarcity (capping upside spikes) and back off during surplus
(avoiding deep negatives). SA1's wind and solar are price-takers:
they produce when the resource is available regardless of price,
creating both the surplus (negative prices at midday) and the
scarcity (evening peak when solar drops). This makes SA1 the
harder — and more valuable — forecasting target.

</details>

In [ ]:
# Your analysis here
sa1 = all_prices["SA1"]["price"].dropna()
tas1 = all_prices["TAS1"]["price"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Price-duration curves
for p, r, c in [(sa1, "SA1", REGION_COLORS["SA1"]),
                (tas1, "TAS1", REGION_COLORS["TAS1"])]:
    sorted_p = np.sort(p.values)[::-1]
    pcts = np.linspace(0, 100, len(sorted_p))
    axes[0].plot(pcts, sorted_p, linewidth=0.8, color=c, label=r)
axes[0].set_yscale("symlog", linthresh=100)
axes[0].set_xlabel("% of time at or above")
axes[0].set_ylabel("Price ($/MWh)")
axes[0].set_title("Price-duration curves")
axes[0].legend()

# QQ plots
stats.probplot(np.arcsinh(sa1.values), dist="norm", plot=axes[1])
axes[1].set_title("QQ plot — SA1 arcsinh(price)")
stats.probplot(np.arcsinh(tas1.values), dist="norm", plot=axes[2])
axes[2].set_title("QQ plot — TAS1 arcsinh(price)")

fig.tight_layout()
plt.show()

for r, p in [("SA1", sa1), ("TAS1", tas1)]:
    print(f"{r}: kurtosis={p.kurtosis():.0f}, skew={p.skew():.1f}, "
          f"{(p < 0).mean():.2%} negative, "
          f"{(p > spike_threshold).mean():.3%} spikes")


---
## What we learned

1. The price distribution has extreme kurtosis — thousands of times
   the Gaussian level. The arcsinh transform helps but doesn't fully
   normalise it.
2. Strong diurnal seasonality driven by the solar duck curve: midday
   dip, evening peak.
3. Lag-48 (same time yesterday) is the single strongest autocorrelation,
   with a secondary peak at lag-336 (same time last week).
4. Volatility clusters: quiet and volatile periods alternate, and
   today's volatility predicts tomorrow's.
5. The price-duration curve shows that >90% of time is spent below
   \$150, but the remaining <10% contains most of the economic action.
6. Regional generation mix matters: SA1 (wind/solar) has heavier tails
   than TAS1 (hydro).
7. Any model must reproduce these stylised facts — a forecast that
   never predicts negative prices or underestimates spike frequency
   is missing the point.

**Next:** Notebook 03 connects these price patterns to their physical
causes — generation mix, net load, and the supply stack.

In [ ]:
# Write report
report_dir = Path(cfg["paths"]["reports"])
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 02 — Market Shape Report

Region: {REGION} | Window: {START} to {END}

## Stylised facts

1. **Heavy tails** — excess kurtosis = {kurt:.0f} (Gaussian = 0).
2. **Diurnal seasonality** — evening peak (17–20h), midday solar dip.
3. **Lag-48 autocorrelation** — same-time-yesterday is strongest predictor.
4. **Volatility clustering** — absolute returns autocorrelated for days.
5. **Asymmetric tails** — right tail (spikes) heavier than left (negatives).
6. **Regional heterogeneity** — SA1 most volatile, TAS1 least.
7. **Non-Gaussian** — 5-sigma events occur orders of magnitude more often than expected.

## Key statistics

- Skewness: {skew:.2f}
- Excess kurtosis: {kurt:.0f}
- Hill tail index (upper): α = {hill_alpha:.2f}
- Price below $50: {(price < 50).mean():.1%} of the time
- Price above ${spike_threshold}: {(price > spike_threshold).mean():.2%}
"""

(report_dir / "02_market_shape.md").write_text(report)
print("Report written to", report_dir / "02_market_shape.md")